# Train the time-series anomaly-detection autoencoder

This notebook trains the Conv1D autoencoder of
[docs/autoencoder.md](https://github.com/mstfaatf/edgebench/blob/main/docs/autoencoder.md)
on the NAB `machine_temperature_system_failure` series and exports it to ONNX.
It runs in Google Colab because TensorFlow has no native Windows on Arm build;
everything after the export (INT8 quantization, NPU checks, benchmarks) happens on
the target machine.

* **Runtime:** CPU (Runtime → Change runtime type → CPU). Training takes a few minutes.
* **Data:** the notebook clones the edgebench repository at a pinned commit and uses
  its own data code (`scripts/get_data.py --dataset nab`, `scripts/common/timeseries.py`),
  so the windows, splits and normalization are exactly those of the committed split
  manifest; the notebook stops if anything differs.
* **Architecture:** built from the layer list in `scripts/common/autoencoder.py`,
  the same list the NPU op-support check used.
* **Reproducibility:** fixed seeds, TensorFlow op determinism and single-threaded
  CPU execution make a rerun on the same Colab machine type repeat the training.
  Across machines (other CPUs, other library builds) it is reproducible only to
  within floating-point noise, not bit for bit; the exported ONNX files and their
  SHA256 in the training record are what the rest of the pipeline uses.
* **Output:** a zip file with the FP32 ONNX files (batch 1, 4 and 8), a training
  record and reference outputs, downloaded at the end. Its SHA256s are printed.

Run all cells in order (Runtime → Run all).

In [ ]:
# Pinned versions. Installed first, before anything imports them.
import subprocess, sys

PINS = {
    "tensorflow": "2.21.0",
    "keras": "3.15.1",
    "tf2onnx": "1.17.0",
    "onnx": "1.23.2",
    "onnxruntime": "1.32.0",
}
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
                *(f"{name}=={version}" for name, version in PINS.items())], check=True)
print(sys.version)

In [ ]:
# Parameters.
REPO_URL = "https://github.com/mstfaatf/edgebench.git"
EDGEBENCH_COMMIT = "1e9eb1bea81f286a8b00574b55f1dfe874ffd80e"  # contains the data code and split manifest used here
REPO_DIR = "edgebench"

DESIGN = "conv1d"              # see scripts/common/autoencoder.py
MODEL_NAME = "conv1d_autoencoder"
SEED = 20261124
BATCH_SIZES = (1, 4, 8)
OPSET = 17

LEARNING_RATE = 1e-3
TRAIN_BATCH = 64
MAX_EPOCHS = 200
PATIENCE = 10                  # early stopping on validation reconstruction error

In [ ]:
# The repository at the pinned commit, and the NAB data checked against its split manifest.
import os, pathlib

if not pathlib.Path(REPO_DIR, ".git").is_dir():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", "--detach", EDGEBENCH_COMMIT], check=True)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], check=True,
                      capture_output=True, text=True).stdout.strip()
assert head == EDGEBENCH_COMMIT, (head, EDGEBENCH_COMMIT)
print("edgebench at", head)

# Downloads the two NAB files from the pinned NAB commit, checks their SHA256 and
# compares the windows and splits with datasets/nab-machine-temperature/splits.json.
subprocess.run([sys.executable, f"{REPO_DIR}/scripts/get_data.py", "--dataset", "nab"], check=True)

In [ ]:
# Windows of every split, rebuilt by the repository's own code (raises if anything
# differs from the committed manifest).
sys.path.insert(0, os.path.abspath(f"{REPO_DIR}/scripts"))
import hashlib, json
import numpy as np
from common import autoencoder as ae
from common import timeseries as ts

data = ts.load_split_windows()
manifest = data.manifest
manifest_sha256 = hashlib.sha256(ts.SPLIT_MANIFEST.read_bytes()).hexdigest()
x_train = ae.model_inputs(DESIGN, data.windows["train"])
x_val = ae.model_inputs(DESIGN, data.windows["validation"])
x_ref = ae.model_inputs(DESIGN, data.reference)
print("split manifest sha256", manifest_sha256)
print("normalization mean", manifest["normalization"]["mean"], "std", manifest["normalization"]["std"])
for name, w in data.windows.items():
    print(f"{name:10} {w.shape[0]:6} windows, {int(data.labels[name].sum())} anomalous")
print("training input", x_train.shape, x_train.dtype, "reference windows", x_ref.shape)

In [ ]:
# TensorFlow with fixed seeds, op determinism and one CPU thread.
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf

# Before anything starts the TensorFlow runtime.
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()

import keras
import onnx
import onnxruntime as ort
import tf2onnx

keras.utils.set_random_seed(SEED)

versions = {"python": sys.version.split()[0], "tensorflow": tf.__version__,
            "keras": keras.__version__, "tf2onnx": tf2onnx.__version__,
            "onnx": onnx.__version__, "onnxruntime": ort.__version__, "numpy": np.__version__}
import google.protobuf
versions["protobuf"] = google.protobuf.__version__
mismatched = {k: (versions[k], v) for k, v in PINS.items() if versions[k] != v}
assert not mismatched, f"installed versions differ from the pins: {mismatched}; restart the session and run all cells again"
assert not tf.config.list_physical_devices("GPU"), "use a CPU runtime"
print(versions)

In [ ]:
# The model, from the shared layer list.
def build_model(design: str) -> keras.Model:
    inputs = keras.Input(shape=tuple(ae.input_shape(design, 1)[1:]), name=ae.INPUT_NAME)
    x = inputs
    for i, layer in enumerate(ae.DESIGNS[design]):
        if layer.kind == "conv":
            x = keras.layers.Conv1D(layer.units, layer.kernel, strides=layer.stride, padding="same",
                                    activation=layer.activation, name=f"conv_{i}")(x)
        elif layer.kind == "conv_transpose":
            x = keras.layers.Conv1DTranspose(layer.units, layer.kernel, strides=layer.stride,
                                             padding="same", activation=layer.activation,
                                             name=f"conv_transpose_{i}")(x)
        else:
            x = keras.layers.Dense(layer.units, activation=layer.activation, name=f"dense_{i}")(x)
    return keras.Model(inputs, x, name=MODEL_NAME)

model = build_model(DESIGN)
model.summary()
assert model.count_params() == ae.parameter_count(DESIGN), (model.count_params(), ae.parameter_count(DESIGN))
assert model.output_shape[1:] == x_train.shape[1:], model.output_shape

In [ ]:
# Training: reconstruction MSE, early stopping on the validation split.
import time

model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse")
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                      restore_best_weights=True, verbose=1)
start = time.perf_counter()
history = model.fit(x_train, x_train, validation_data=(x_val, x_val), epochs=MAX_EPOCHS,
                    batch_size=TRAIN_BATCH, shuffle=True, callbacks=[early], verbose=2)
train_seconds = time.perf_counter() - start
curve = [{"epoch": i + 1, "loss": float(l), "val_loss": float(v)}
         for i, (l, v) in enumerate(zip(history.history["loss"], history.history["val_loss"]))]
best = min(curve, key=lambda c: c["val_loss"])
print(f"{len(curve)} epochs in {train_seconds:.0f} s; best epoch {best['epoch']} "
      f"(val_loss {best['val_loss']:.6f})")

In [ ]:
# Training curve and final reconstruction errors (weights of the best epoch).
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot([c["epoch"] for c in curve], [c["loss"] for c in curve], label="training")
plt.plot([c["epoch"] for c in curve], [c["val_loss"] for c in curve], label="validation")
plt.axvline(best["epoch"], color="grey", linestyle=":", label="best epoch")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("reconstruction MSE"); plt.legend(); plt.show()

final_train_loss = float(np.ravel(model.evaluate(x_train, x_train, batch_size=256, verbose=0))[0])
final_val_loss = float(np.ravel(model.evaluate(x_val, x_val, batch_size=256, verbose=0))[0])
val_errors = np.mean((model.predict(x_val, batch_size=256, verbose=0) - x_val) ** 2,
                     axis=tuple(range(1, x_val.ndim)))
print(f"final training MSE {final_train_loss:.6f}, validation MSE {final_val_loss:.6f}")
print("validation per-window MSE percentiles 50/99/100:",
      np.percentile(val_errors, [50, 99, 100]).round(6))

In [ ]:
# ONNX export with tf2onnx: one file per batch size, fixed input shape, opset 17,
# input "input" and output "reconstruction"; then onnxruntime (CPU) against Keras.
from collections import Counter
from common.onnx_export import dims, static_shape_problems

BUNDLE = f"{MODEL_NAME}_colab_bundle_{EDGEBENCH_COMMIT[:7]}"
bundle = pathlib.Path(BUNDLE)
bundle.mkdir(exist_ok=True)

keras_val = model.predict(x_val, batch_size=256, verbose=0)
keras_ref = model(x_ref, training=False).numpy()

def run_ort(path, x, batch):
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    n = len(x) // batch * batch  # whole batches only
    return np.concatenate([session.run(None, {ae.INPUT_NAME: x[i:i + batch]})[0]
                           for i in range(0, n, batch)])

exports = {}
ort_outputs = {}
for batch in BATCH_SIZES:
    spec = (tf.TensorSpec(ae.input_shape(DESIGN, batch), tf.float32, name=ae.INPUT_NAME),)
    serve = tf.function(lambda x: model(x, training=False))
    proto, _ = tf2onnx.convert.from_function(serve, input_signature=spec, opset=OPSET)
    ae.rename_io(proto)
    proto = onnx.shape_inference.infer_shapes(proto, strict_mode=True)
    onnx.checker.check_model(proto, full_check=True)
    problems = static_shape_problems(proto) + [
        f"value_info '{v.name}' has shape {dims(v)}" for v in proto.graph.value_info
        if not all(isinstance(d, int) and d > 0 for d in dims(v))]
    assert not problems, problems
    path = bundle / f"{MODEL_NAME}_fp32_b{batch}.onnx"
    onnx.save(proto, str(path))
    out = run_ort(path, x_val, batch)
    ort_outputs[batch] = out
    diff = float(np.abs(out - keras_val[:len(out)]).max())
    ref_diff = float(np.abs(run_ort(path, x_ref, batch) - keras_ref[:len(x_ref) // batch * batch]).max())
    exports[batch] = {
        "file": path.name,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "size_bytes": path.stat().st_size,
        "opset": next(o.version for o in proto.opset_import if o.domain in ("", "ai.onnx")),
        "ir_version": proto.ir_version,
        "producer": f"{proto.producer_name} {proto.producer_version}",
        "input_shape": ae.input_shape(DESIGN, batch),
        "ops": dict(sorted(Counter(n.op_type for n in proto.graph.node).items())),
        "nodes": len(proto.graph.node),
        "keras_vs_onnx_max_abs_validation": diff,
        "keras_vs_onnx_max_abs_reference": ref_diff,
        "compared_windows": int(len(out)),
    }
    print(f"b{batch}: {path.name} {exports[batch]['size_bytes']} B, {exports[batch]['nodes']} nodes "
          f"{exports[batch]['ops']}\n    Keras vs ONNX max abs diff {diff:.3e} "
          f"({len(out)} validation windows), {ref_diff:.3e} (reference windows)")
for batch in BATCH_SIZES[1:]:
    n = len(ort_outputs[batch])
    d = float(np.abs(ort_outputs[batch] - ort_outputs[1][:n]).max())
    exports[batch]["onnx_vs_b1_max_abs"] = d
    print(f"ONNX b{batch} vs b1 max abs diff {d:.3e}")

In [ ]:
# Reference outputs on the fixed test windows of the split manifest.
ort_ref = run_ort(bundle / f"{MODEL_NAME}_fp32_b1.onnx", x_ref, 1)
np.save(bundle / "reference_inputs.npy", x_ref)
np.save(bundle / "reference_keras.npy", keras_ref)
np.save(bundle / "reference_onnx_b1.npy", ort_ref)
print("reference windows (start index):", manifest["reference_windows"]["window_starts"])
print("Keras vs ONNX b1 on them:", float(np.abs(ort_ref - keras_ref).max()))

In [ ]:
# Training record.
import platform, datetime

def sha256(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()

summary_lines = []
model.summary(print_fn=lambda line, *args, **kwargs: summary_lines.append(line))
layers = [{"kind": l.kind, "units": l.units, "kernel": l.kernel, "stride": l.stride,
           "activation": l.activation} for l in ae.DESIGNS[DESIGN]]
record = {
    "record_version": "1.0.0",
    "model": MODEL_NAME,
    "design": DESIGN,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "edgebench_commit": EDGEBENCH_COMMIT,
    "versions": versions,
    "machine": {"platform": platform.platform(), "processor": platform.processor(),
                "cpu_count": os.cpu_count()},
    "reproducibility": {
        "seed": SEED,
        "keras_set_random_seed": True,
        "tf_op_determinism": True,
        "intra_op_threads": 1,
        "inter_op_threads": 1,
        "note": "Repeatable on the same machine type and library builds; across machines "
                "only to within floating-point noise, not bit for bit.",
    },
    "data": {
        "split_manifest": "datasets/nab-machine-temperature/splits.json",
        "split_manifest_sha256": manifest_sha256,
        "series_sha256": manifest["source"]["series"]["sha256"],
        "labels_sha256": manifest["source"]["labels"]["sha256"],
        "normalized_float32_sha256": manifest["normalization"]["normalized_float32_sha256"],
        "normalization": {k: manifest["normalization"][k] for k in ("formula", "mean", "std", "std_ddof")},
        "window_length": manifest["windowing"]["length"],
        "window_stride": manifest["windowing"]["stride"],
        "train_windows": int(len(x_train)),
        "validation_windows": int(len(x_val)),
    },
    "architecture": {
        "layers": layers,
        "parameters": int(model.count_params()),
        "input_shape": list(model.input_shape),
        "output_shape": list(model.output_shape),
        "summary": summary_lines,
    },
    "training": {
        "loss": "mse",
        "optimizer": "adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": TRAIN_BATCH,
        "max_epochs": MAX_EPOCHS,
        "early_stopping": {"monitor": "val_loss", "patience": PATIENCE, "restore_best_weights": True},
        "shuffle": "training windows, every epoch (seeded)",
        "epochs_run": len(curve),
        "best_epoch": best["epoch"],
        "seconds": round(train_seconds, 1),
        "curve": curve,
        "final_train_mse": final_train_loss,
        "final_validation_mse": final_val_loss,
        "validation_window_mse_percentiles": dict(zip(("p50", "p99", "max"),
                                                      map(float, np.percentile(val_errors, [50, 99, 100])))),
    },
    "export": {
        "tool": "tf2onnx.convert.from_function",
        "opset": OPSET,
        "input_name": ae.INPUT_NAME,
        "output_name": ae.OUTPUT_NAME,
        "files": {f"b{b}": e for b, e in exports.items()},
    },
    "reference_outputs": {
        "window_starts": manifest["reference_windows"]["window_starts"],
        "files": {name: sha256(bundle / name) for name in
                  ("reference_inputs.npy", "reference_keras.npy", "reference_onnx_b1.npy")},
        "keras_vs_onnx_b1_max_abs": float(np.abs(ort_ref - keras_ref).max()),
    },
}
(bundle / "training_record.json").write_text(json.dumps(record, indent=2) + "\n")
print(json.dumps({k: record[k] for k in ("versions", "export")}, indent=2)[:3000])

In [ ]:
# The bundle: SHA256SUMS, a zip file, and the download.
import zipfile

files = sorted(p for p in bundle.iterdir() if p.is_file() and p.name != "SHA256SUMS")
(bundle / "SHA256SUMS").write_text("".join(f"{sha256(p)}  {p.name}\n" for p in files))
zip_path = pathlib.Path(f"{BUNDLE}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(bundle.iterdir()):
        z.write(p, f"{BUNDLE}/{p.name}")
print((bundle / "SHA256SUMS").read_text())
print(f"{zip_path}  sha256 {sha256(zip_path)}  ({zip_path.stat().st_size} bytes)")
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("not running in Colab: the zip file is in", zip_path.resolve())

## Where the bundle goes

Unzip the downloaded file into `models/colab/` of your edgebench checkout, so that
the files end up in `models/colab/conv1d_autoencoder_colab_bundle_<commit>/`
(`models/` is not committed). Then bring the model through the pipeline on the
target machine; the SHA256s printed above are recorded in `training_record.json`
and `SHA256SUMS` and are checked there.